# Exploratory Data Analysis - Rossmann Store Sales

This notebook explores the Rossmann Store Sales dataset to understand patterns, distributions, and relationships.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.config import load_config
from src.data.loader import load_raw_data

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

config = load_config()
df, store_df = load_raw_data(config)
print(f'Shape: {df.shape}')
df.head()

## 1. Summary Statistics & Missing Values

In [ ]:
print('=== Summary Statistics ===')
display(df.describe())
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\n=== Data Types ===')
print(df.dtypes)

## 2. Sales Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall sales distribution
axes[0].hist(df['Sales'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Overall Sales Distribution')
axes[0].set_xlabel('Sales')

# By Store Type
for st in sorted(df['StoreType'].unique()):
    subset = df[df['StoreType'] == st]['Sales']
    axes[1].hist(subset, bins=30, alpha=0.5, label=f'Type {st}')
axes[1].set_title('Sales by Store Type')
axes[1].legend()

# By Assortment
for a in sorted(df['Assortment'].unique()):
    subset = df[df['Assortment'] == a]['Sales']
    axes[2].hist(subset, bins=30, alpha=0.5, label=f'Assortment {a}')
axes[2].set_title('Sales by Assortment')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Time Series Trends

In [ ]:
daily_avg = df.groupby('Date')['Sales'].mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Daily average sales
axes[0].plot(daily_avg.index, daily_avg.values, linewidth=0.5)
axes[0].set_title('Daily Average Sales (All Stores)')
axes[0].set_ylabel('Average Sales')

# Monthly average
monthly = df.groupby(df['Date'].dt.to_period('M'))['Sales'].mean()
axes[1].bar(range(len(monthly)), monthly.values)
axes[1].set_title('Monthly Average Sales')
axes[1].set_xticks(range(0, len(monthly), 3))
axes[1].set_xticklabels([str(p) for p in monthly.index[::3]], rotation=45)

plt.tight_layout()
plt.show()

## 4. Weekly Seasonality

In [ ]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_sales = df.groupby('DayOfWeek')['Sales'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(1, 8), [dow_sales.get(i, 0) for i in range(1, 8)])
ax.set_xticks(range(1, 8))
ax.set_xticklabels(day_names)
ax.set_title('Average Sales by Day of Week')
ax.set_ylabel('Average Sales')
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Sales vs Customers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(df['Customers'], df['Sales'], alpha=0.01, s=1)
axes[0].set_xlabel('Customers')
axes[0].set_ylabel('Sales')
axes[0].set_title('Sales vs Customers')

# Promo effect
promo_sales = df.groupby('Promo')['Sales'].mean()
axes[1].bar(['No Promo', 'Promo'], promo_sales.values)
axes[1].set_title('Average Sales: Promo vs No Promo')

# Competition distance effect
df['CompDistBin'] = pd.cut(df['CompetitionDistance'], bins=10)
comp_sales = df.groupby('CompDistBin', observed=True)['Sales'].mean()
axes[2].bar(range(len(comp_sales)), comp_sales.values)
axes[2].set_title('Sales by Competition Distance')
axes[2].set_xticks([])

plt.tight_layout()
plt.show()

## 6. Holiday Effects

In [ ]:
holiday_sales = df.groupby('StateHoliday')['Sales'].agg(['mean', 'count'])
print('Sales by State Holiday type:')
display(holiday_sales)

school_sales = df.groupby('SchoolHoliday')['Sales'].mean()
print(f'\nSchool Holiday effect: No={school_sales[0]:.0f}, Yes={school_sales[1]:.0f}')